In [10]:
import json
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, accuracy_score

BASE_DIR = Path.cwd().parent.parent.parent.parent

OUTPUT_DIR1 = BASE_DIR / "outputs"
SPLIT_DIR = OUTPUT_DIR1 / "image_modeling"

RUN_NAME = "mm_camembert_clip_gated_fusion_staged_unfreeze"
MULTI_MODEL_PATH = SPLIT_DIR / "multimodal"
MODEL_OUTPUT_DIR = MULTI_MODEL_PATH / RUN_NAME
OUTPUT_DIR = MULTI_MODEL_PATH / RUN_NAME / 'firstfusion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LAST_CHECKPOINT_PATH = OUTPUT_DIR / "last_checkpoint.pt"
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "best_checkpoint.pt"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model_state_dict.pt"
BEST_VAL_LOGITS_PATH = MODEL_OUTPUT_DIR / "best_val_logits.npy"
HISTORY_CSV_PATH = OUTPUT_DIR / "training_history.csv"

TEXT_PATH = OUTPUT_DIR1 / "y_logits_camembert_run2_epochs3.npy"

STACK_MODEL_PATH = OUTPUT_DIR / "final_stacking_model.pkl"
STACK_FEATURES_PATH = OUTPUT_DIR / "stacking_features.npy"
STACK_PRED_PATH = OUTPUT_DIR / "stacking_predictions.npy"
STACK_REPORT_PATH = OUTPUT_DIR / "stacking_report.csv"
STACK_METADATA_PATH = OUTPUT_DIR / "stacking_metadata.json"
STACK_PROBA_PATH = OUTPUT_DIR / "stacking_probabilities.npy"

multi_logits = np.load(BEST_VAL_LOGITS_PATH)
text_logits = np.load(TEXT_PATH)

print("multimodal logits shape:", multi_logits.shape)
print("text logits shape      :", text_logits.shape)

if multi_logits.shape != text_logits.shape:
    raise ValueError(
        f"Shape mismatch: multimodal={multi_logits.shape}, text={text_logits.shape}"
    )

val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json", "r") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

y_true = val_df["prdtypecode"].map(label2id).to_numpy()

if len(y_true) != multi_logits.shape[0]:
    raise ValueError(
        f"Validation length mismatch: y_true={len(y_true)}, logits={multi_logits.shape[0]}"
    )

# ============================================================
# Stacking input features
# ============================================================
# Burada iki modelin logitslerini yan yana koyuyoruz.
# 27 class + 27 class = 54 feature
X = np.concatenate([multi_logits, text_logits], axis=1)

print("stacking feature shape :", X.shape)

# ============================================================
# Final stacking model
# ============================================================
stack_model = LogisticRegression(
    max_iter=3000,
    solver="lbfgs"
)

stack_model.fit(X, y_true)

pred = stack_model.predict(X)
proba = stack_model.predict_proba(X)

macro_f1 = f1_score(y_true, pred, average="macro")
acc = accuracy_score(y_true, pred)

print("\nFINAL STACKING RESULTS")
print("Accuracy :", round(acc, 6))
print("Macro F1 :", round(macro_f1, 6))

# ============================================================
# Save artifacts
# ============================================================
joblib.dump(stack_model, STACK_MODEL_PATH)
np.save(STACK_FEATURES_PATH, X)
np.save(STACK_PRED_PATH, pred)
np.save(STACK_PROBA_PATH, proba)

report_df = pd.DataFrame(
    classification_report(y_true, pred, output_dict=True, zero_division=0)
).transpose()
report_df.to_csv(STACK_REPORT_PATH)

with open(STACK_METADATA_PATH, "w") as f:
    json.dump(
        {
            "model_type": "LogisticRegression_stacking",
            "macro_f1": float(macro_f1),
            "accuracy": float(acc),
            "multimodal_shape": list(multi_logits.shape),
            "text_shape": list(text_logits.shape),
            "stacking_feature_shape": list(X.shape),
        },
        f,
        indent=2
    )

print("\nSaved:")
print(STACK_MODEL_PATH)
print(STACK_FEATURES_PATH)
print(STACK_PRED_PATH)
print(STACK_PROBA_PATH)
print(STACK_REPORT_PATH)
print(STACK_METADATA_PATH)

multimodal logits shape: (16984, 27)
text logits shape      : (16984, 27)
stacking feature shape : (16984, 54)

FINAL STACKING RESULTS
Accuracy : 0.919748
Macro F1 : 0.91127

Saved:
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusion_staged_unfreeze/firstfusion/final_stacking_model.pkl
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusion_staged_unfreeze/firstfusion/stacking_features.npy
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusion_staged_unfreeze/firstfusion/stacking_predictions.npy
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusion_staged_unfreeze/firstfusion/stacking_probabilities.npy
/Users/sumeyraozdemir/D

In [7]:
import json
from pathlib import Path
import joblib

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, accuracy_score

BASE_DIR = Path.cwd().parent.parent.parent.parent

OUTPUT_DIR1 = BASE_DIR / "outputs"
SPLIT_DIR = OUTPUT_DIR1 / "image_modeling"

RUN_NAME = "mm_camembert_clip_gated_fusion_staged_unfreeze"
TEXT_NPY_DIR = BASE_DIR / "outputs"
OUTPUT_DIR = BASE_DIR / "outputs" / "image_modeling" / "multimodal" / RUN_NAME / "fusion"
MODEL_DIR = BASE_DIR / "outputs" / "image_modeling" / "multimodal" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_VAL_LOGITS_PATH = MODEL_DIR / "best_val_logits.npy"
TEXT_PATH = TEXT_NPY_DIR / "y_logits_camembert_run2_epochs3.npy"

STACK_MODEL_PATH = OUTPUT_DIR / "final_stacking_model_split.pkl"
STACK_FEATURES_TRAIN_PATH = OUTPUT_DIR / "stacking_X_meta_train.npy"
STACK_FEATURES_VAL_PATH = OUTPUT_DIR / "stacking_X_meta_val.npy"
STACK_Y_TRAIN_PATH = OUTPUT_DIR / "stacking_y_meta_train.npy"
STACK_Y_VAL_PATH = OUTPUT_DIR / "stacking_y_meta_val.npy"
STACK_PRED_PATH = OUTPUT_DIR / "stacking_predictions_meta_val.npy"
STACK_PROBA_PATH = OUTPUT_DIR / "stacking_probabilities_meta_val.npy"
STACK_REPORT_PATH = OUTPUT_DIR / "stacking_report_meta_val.csv"
STACK_METADATA_PATH = OUTPUT_DIR / "stacking_metadata_meta_val.json"

multi_logits = np.load(BEST_VAL_LOGITS_PATH)
text_logits = np.load(TEXT_PATH)

print("multimodal logits shape:", multi_logits.shape)
print("text logits shape      :", text_logits.shape)

if multi_logits.shape != text_logits.shape:
    raise ValueError(
        f"Shape mismatch: multimodal={multi_logits.shape}, text={text_logits.shape}"
    )

val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json", "r", encoding="utf-8") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

y_true = val_df["prdtypecode"].map(label2id).to_numpy()

if len(y_true) != multi_logits.shape[0]:
    raise ValueError(
        f"Validation length mismatch: y_true={len(y_true)}, logits={multi_logits.shape[0]}"
    )

X = np.concatenate([multi_logits, text_logits], axis=1)

print("stacking feature shape:", X.shape)

X_meta_train, X_meta_val, y_meta_train, y_meta_val = train_test_split(
    X,
    y_true,
    test_size=0.3,
    random_state=42,
    stratify=y_true
)

print("X_meta_train:", X_meta_train.shape)
print("X_meta_val  :", X_meta_val.shape)

stack_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        C=1.0
    ))
])

stack_model.fit(X_meta_train, y_meta_train)

pred = stack_model.predict(X_meta_val)
proba = stack_model.predict_proba(X_meta_val)

macro_f1 = f1_score(y_meta_val, pred, average="macro")
weighted_f1 = f1_score(y_meta_val, pred, average="weighted")
acc = accuracy_score(y_meta_val, pred)

print("\nSTACKING RESULTS ON META-VAL")
print("Accuracy    :", round(acc, 6))
print("Macro F1    :", round(macro_f1, 6))
print("Weighted F1 :", round(weighted_f1, 6))

report_df = pd.DataFrame(
    classification_report(y_meta_val, pred, output_dict=True, zero_division=0)
).transpose()

joblib.dump(stack_model, STACK_MODEL_PATH)
np.save(STACK_FEATURES_TRAIN_PATH, X_meta_train)
np.save(STACK_FEATURES_VAL_PATH, X_meta_val)
np.save(STACK_Y_TRAIN_PATH, y_meta_train)
np.save(STACK_Y_VAL_PATH, y_meta_val)
np.save(STACK_PRED_PATH, pred)
np.save(STACK_PROBA_PATH, proba)
report_df.to_csv(STACK_REPORT_PATH)

with open(STACK_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model_type": "LogisticRegression_stacking_meta_split",
            "accuracy": float(acc),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "multimodal_shape": list(multi_logits.shape),
            "text_shape": list(text_logits.shape),
            "stacking_feature_shape": list(X.shape),
            "X_meta_train_shape": list(X_meta_train.shape),
            "X_meta_val_shape": list(X_meta_val.shape),
            "test_size": 0.3,
            "random_state": 42
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("\nSaved:")
print(STACK_MODEL_PATH)
print(STACK_FEATURES_TRAIN_PATH)
print(STACK_FEATURES_VAL_PATH)
print(STACK_Y_TRAIN_PATH)
print(STACK_Y_VAL_PATH)
print(STACK_PRED_PATH)
print(STACK_PROBA_PATH)
print(STACK_REPORT_PATH)
print(STACK_METADATA_PATH)

multimodal logits shape: (16984, 27)
text logits shape      : (16984, 27)
stacking feature shape: (16984, 54)
X_meta_train: (11888, 54)
X_meta_val  : (5096, 54)

STACKING RESULTS ON META-VAL
Accuracy    : 0.898548
Macro F1    : 0.883834
Weighted F1 : 0.898054

Saved:
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusion_staged_unfreeze/fusion/final_stacking_model_split.pkl
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusion_staged_unfreeze/fusion/stacking_X_meta_train.npy
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusion_staged_unfreeze/fusion/stacking_X_meta_val.npy
/Users/sumeyraozdemir/Documents/DS_Projects/rakuten-project/Rakuten_Data_Science/outputs/image_modeling/multimodel/mm_camembert_clip_gated_fusio

In [ ]:
import json
from pathlib import Path
import joblib
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoTokenizer, CamembertModel, CLIPProcessor, CLIPVisionModel

BASE_DIR = Path.cwd().parent.parent.parent.parent

OUTPUT_DIR1 = BASE_DIR / "outputs"
SPLIT_DIR = OUTPUT_DIR1 / "image_modeling"

RUN_NAME = "mm_camembert_clip_gated_fusion_staged_unfreeze"
MODEL_DIR = BASE_DIR / "outputs" / "multimodal" / RUN_NAME
OUTPUT_DIR = BASE_DIR / "outputs" / "multimodal" / RUN_NAME / "for_prediction"

MULTI_CHECKPOINT_PATH = MODEL_DIR / "best_checkpoint.pt"
STACK_MODEL_PATH = MODEL_DIR / "fusion" / "final_stacking_model_split.pkl"

TEXT_MODEL_DIR = OUTPUT_DIR1 / "camembert_run2_epochs3"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"
MAX_TEXT_LEN = 128

with open(SPLIT_DIR / "label2id.json", "r") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}

id2label = {v: k for k, v in label2id.items()}
num_classes = len(label2id)

tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)


class CamembertTextClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.encoder = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        return self.classifier(cls)


class MultimodalGatedFusionModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.text_encoder = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision_encoder = CLIPVisionModel.from_pretrained(CLIP_ID)

        self.text_proj = nn.Linear(self.text_encoder.config.hidden_size, 768)
        self.vision_proj = nn.Linear(self.vision_encoder.config.hidden_size, 768)

        self.gate = nn.Sequential(
            nn.Linear(768 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 768),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(768 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        t = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        v = self.vision_encoder(pixel_values=pixel_values).pooler_output

        t = self.text_proj(t)
        v = self.vision_proj(v)

        t = nn.functional.normalize(t, dim=1)
        v = nn.functional.normalize(v, dim=1)

        fusion = torch.cat([t, v], dim=1)
        g = self.gate(fusion)

        fused = g * v + (1 - g) * t
        final = torch.cat([fused, t, v], dim=1)

        return self.classifier(final)


def load_multimodal():
    ckpt = torch.load(MULTI_CHECKPOINT_PATH, map_location=DEVICE)
    model = MultimodalGatedFusionModel(num_classes)

    state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state)

    model.to(DEVICE).eval()
    return model


def load_text_model():
    model = CamembertTextClassifier(num_classes)

    path1 = TEXT_MODEL_DIR / "best_model_state_dict.pt"
    path2 = TEXT_MODEL_DIR / "best_checkpoint.pt"

    if path1.exists():
        state = torch.load(path1, map_location=DEVICE)
    else:
        ckpt = torch.load(path2, map_location=DEVICE)
        state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model


def prepare_text(designation, description):
    text = (designation + " " + description).strip()

    if text == "":
        text = "unknown product"

    enc = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LEN,
        return_tensors="pt"
    )

    return {
        "input_ids": enc["input_ids"].to(DEVICE),
        "attention_mask": enc["attention_mask"].to(DEVICE)
    }


def prepare_image(path):
    img = Image.open(path).convert("RGB")
    return clip_processor(images=img, return_tensors="pt")["pixel_values"].to(DEVICE)


@torch.no_grad()
def predict_single(image_path, designation="", description=""):
    multi_model = load_multimodal()
    text_model = load_text_model()
    stack_model = joblib.load(STACK_MODEL_PATH)

    text_inputs = prepare_text(designation, description)
    pixel_values = prepare_image(image_path)

    multi_logits = multi_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"],
        pixel_values=pixel_values
    ).cpu().numpy()

    text_logits = text_model(
        input_ids=text_inputs["input_ids"],
        attention_mask=text_inputs["attention_mask"]
    ).cpu().numpy()

    X = np.concatenate([multi_logits, text_logits], axis=1)

    pred = int(stack_model.predict(X)[0])
    proba = stack_model.predict_proba(X)[0]

    return {
        "class_id": pred,
        "prdtypecode": int(id2label[pred]),
        "confidence": float(proba[pred]),
        "top3": np.argsort(proba)[::-1][:3]
    }